In [1]:
!pip install timm ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 74.2 MB/s eta 0:00:00


In [2]:
import os, zipfile, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Device: cuda
Tesla T4


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
ZIP_PATH = "/content/drive/MyDrive/scenario23_dev_w_resources.zip"
EXTRACT_ROOT = "/content/scenario23_dev_w_resources"

os.makedirs(EXTRACT_ROOT, exist_ok=True)

if not os.path.exists(os.path.join(EXTRACT_ROOT, "scenario23_dev")):
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_ROOT)

SCENARIO_CSV = "/content/scenario23_dev_w_resources/scenario23_dev/scenario23.csv"

df = pd.read_csv(SCENARIO_CSV)
df = df.sort_values(["seq_index", "index"]).reset_index(drop=True)

print(df.shape)
print(df.columns.tolist())
df.head()

(11387, 17)
['index', 'unit1_rgb', 'unit1_pwr_60ghz', 'unit1_loc', 'unit2_loc', 'unit2_speed', 'unit2_altitude', 'unit2_distance', 'unit2_height', 'unit2_x-speed', 'unit2_y-speed', 'unit2_z-speed', 'unit2_pitch', 'unit2_roll', 'seq_index', 'time_stamp[UTC]', 'unit1_beam_index']


,index,unit1_rgb,unit1_pwr_60ghz,unit1_loc,unit2_loc,unit2_speed,unit2_altitude,unit2_distance,unit2_height,unit2_x-speed,unit2_y-speed,unit2_z-speed,unit2_pitch,unit2_roll,seq_index,time_stamp[UTC],unit1_beam_index
0,1,./unit1/camera_data/image_BS1_1_16_58_42.jpg,./unit1/mmWave_data/mmWave_power_1.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_1.txt,./unit2/speed/speed_1.txt,./unit2/altitude/altitude_1.txt,./unit2/distance/distance_1.txt,./unit2/height/height_1.txt,./unit2/x_speed/x_speed_1.txt,./unit2/y_speed/y_speed_1.txt,./unit2/z_speed/z_speed_1.txt,./unit2/pitch/pitch_1.txt,./unit2/roll/roll_1.txt,1,['16-58-42-0'],43
1,2,./unit1/camera_data/image_BS1_2_16_58_42.jpg,./unit1/mmWave_data/mmWave_power_2.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_2.txt,./unit2/speed/speed_2.txt,./unit2/altitude/altitude_2.txt,./unit2/distance/distance_2.txt,./unit2/height/height_2.txt,./unit2/x_speed/x_speed_2.txt,./unit2/y_speed/y_speed_2.txt,./unit2/z_speed/z_speed_2.txt,./unit2/pitch/pitch_2.txt,./unit2/roll/roll_2.txt,1,['16-58-42-142'],43
2,3,./unit1/camera_data/image_BS1_3_16_58_42.jpg,./unit1/mmWave_data/mmWave_power_3.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_3.txt,./unit2/speed/speed_3.txt,./unit2/altitude/altitude_3.txt,./unit2/distance/distance_3.txt,./unit2/height/height_3.txt,./unit2/x_speed/x_speed_3.txt,./unit2/y_speed/y_speed_3.txt,./unit2/z_speed/z_speed_3.txt,./unit2/pitch/pitch_3.txt,./unit2/roll/roll_3.txt,1,['16-58-42-284'],43
3,4,./unit1/camera_data/image_BS1_4_16_58_42.jpg,./unit1/mmWave_data/mmWave_power_4.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_4.txt,./unit2/speed/speed_4.txt,./unit2/altitude/altitude_4.txt,./unit2/distance/distance_4.txt,./unit2/height/height_4.txt,./unit2/x_speed/x_speed_4.txt,./unit2/y_speed/y_speed_4.txt,./unit2/z_speed/z_speed_4.txt,./unit2/pitch/pitch_4.txt,./unit2/roll/roll_4.txt,1,['16-58-42-426'],43
4,5,./unit1/camera_data/image_BS1_5_16_58_42.jpg,./unit1/mmWave_data/mmWave_power_5.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_5.txt,./unit2/speed/speed_5.txt,./unit2/altitude/altitude_5.txt,./unit2/distance/distance_5.txt,./unit2/height/height_5.txt,./unit2/x_speed/x_speed_5.txt,./unit2/y_speed/y_speed_5.txt,./unit2/z_speed/z_speed_5.txt,./unit2/pitch/pitch_5.txt,./unit2/roll/roll_5.txt,1,['16-58-42-568'],39


In [5]:
# paper-style label: 64 oversampled beam -> 32 downsampled beam
df["beam_32"] = ((df["unit1_beam_index"].astype(int) - 1) // 2).astype(int)

le = LabelEncoder()
df["beam_label"] = le.fit_transform(df["beam_32"])

num_classes = df["beam_label"].nunique()

print("Raw beam unique:", df["unit1_beam_index"].nunique())
print("Raw beam min/max:", df["unit1_beam_index"].min(), df["unit1_beam_index"].max())
print("Downsampled beam_32 unique:", df["beam_32"].nunique())
print("Active model classes:", num_classes)
print("Original beam_32 labels:", sorted(df["beam_32"].unique()))

Raw beam unique: 56
Raw beam min/max: 2 60
Downsampled beam_32 unique: 30
Active model classes: 30
Original beam_32 labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29)]


In [6]:
# image path resolver
all_images = []

for root, dirs, files in os.walk(EXTRACT_ROOT):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
            all_images.append(os.path.join(root, f))

print("Total images:", len(all_images))

image_by_basename = {os.path.basename(p): p for p in all_images}

def resolve_image_path(x):
    x = str(x).replace("\\", "/")
    base = os.path.basename(x)

    if os.path.exists(x):
        return x

    candidate = os.path.join(EXTRACT_ROOT, x.lstrip("./"))
    if os.path.exists(candidate):
        return candidate

    if base in image_by_basename:
        return image_by_basename[base]

    return None

df["image_path"] = df["unit1_rgb"].apply(resolve_image_path)

print("Missing image:", df["image_path"].isna().sum())
assert df["image_path"].isna().sum() == 0

df[["index", "unit1_rgb", "image_path", "unit1_beam_index", "beam_32", "beam_label"]].head()

Total images: 11387
Missing image: 0


,index,unit1_rgb,image_path,unit1_beam_index,beam_32,beam_label
0,1,./unit1/camera_data/image_BS1_1_16_58_42.jpg,/content/scenario23_dev_w_resources/scenario23...,43,21,21
1,2,./unit1/camera_data/image_BS1_2_16_58_42.jpg,/content/scenario23_dev_w_resources/scenario23...,43,21,21
2,3,./unit1/camera_data/image_BS1_3_16_58_42.jpg,/content/scenario23_dev_w_resources/scenario23...,43,21,21
3,4,./unit1/camera_data/image_BS1_4_16_58_42.jpg,/content/scenario23_dev_w_resources/scenario23...,43,21,21
4,5,./unit1/camera_data/image_BS1_5_16_58_42.jpg,/content/scenario23_dev_w_resources/scenario23...,39,19,19


In [7]:
# position feature engineering
pos_cols = [
    "unit2_speed",
    "unit2_altitude",
    "unit2_distance",
    "unit2_height",
    "unit2_x-speed",
    "unit2_y-speed",
    "unit2_z-speed",
    "unit2_pitch",
    "unit2_roll",
]

for c in pos_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df[pos_cols] = df[pos_cols].fillna(df[pos_cols].median())

df["velocity_mag"] = np.sqrt(
    df["unit2_x-speed"]**2 + df["unit2_y-speed"]**2 + df["unit2_z-speed"]**2
)

df["height_distance_ratio"] = df["unit2_height"] / (df["unit2_distance"] + 1e-6)

pos_cols = pos_cols + ["velocity_mag", "height_distance_ratio"]

scaler = StandardScaler()
df[pos_cols] = scaler.fit_transform(df[pos_cols].values)

pos_dim = len(pos_cols)

print("pos_dim:", pos_dim)
print(pos_cols)

pos_dim: 11
['unit2_speed', 'unit2_altitude', 'unit2_distance', 'unit2_height', 'unit2_x-speed', 'unit2_y-speed', 'unit2_z-speed', 'unit2_pitch', 'unit2_roll', 'velocity_mag', 'height_distance_ratio']


In [8]:
# pseudo-swarm: 5 consecutive UAV samples = 5 users
SWARM_SIZE = 5
groups = []

for seq_id, g in df.groupby("seq_index"):
    g = g.sort_values("index").reset_index(drop=True)

    for i in range(0, len(g) - SWARM_SIZE + 1):
        chunk = g.iloc[i:i+SWARM_SIZE]

        img_paths = chunk["image_path"].tolist()
        pos_feats = chunk[pos_cols].values.astype(np.float32)
        labels = chunk["beam_label"].values.astype(np.int64)

        groups.append((img_paths, pos_feats, labels))

print("Pseudo-swarm groups:", len(groups))

train_groups, temp_groups = train_test_split(
    groups,
    test_size=0.30,
    random_state=SEED,
    shuffle=True
)

val_groups, test_groups = train_test_split(
    temp_groups,
    test_size=1/3,
    random_state=SEED,
    shuffle=True
)

print("train:", len(train_groups))
print("val  :", len(val_groups))
print("test :", len(test_groups))

Pseudo-swarm groups: 11255
train: 7878
val  : 2251
test : 1126


In [13]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.25,
            contrast=0.25,
            saturation=0.20,
            hue=0.03
        )
    ], p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class PseudoSwarmDataset(Dataset):
    def __init__(self, groups, transform=None):
        self.groups = groups
        self.transform = transform

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        img_paths, pos_feats, labels = self.groups[idx]

        imgs = []
        for p in img_paths:
            img = Image.open(p).convert("RGB")
            if self.transform:
                img = self.transform(img)
            imgs.append(img)

        imgs = torch.stack(imgs, dim=0)              # [5, 3, 224, 224]
        pos = torch.tensor(pos_feats).float()        # [5, pos_dim]
        y = torch.tensor(labels).long()              # [5]

        return imgs, pos, y


BATCH_SIZE = 8

train_ds = PseudoSwarmDataset(train_groups, train_transform)
val_ds = PseudoSwarmDataset(val_groups, eval_transform)
test_ds = PseudoSwarmDataset(test_groups, eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

imgs_b, pos_b, y_b = next(iter(train_loader))
print("imgs:", imgs_b.shape)
print("pos :", pos_b.shape)
print("y   :", y_b.shape)

imgs: torch.Size([8, 5, 3, 224, 224])
pos : torch.Size([8, 5, 11])
y   : torch.Size([8, 5])


In [14]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False)
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        attn = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(attn))


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x

In [15]:
class ConvNeXtCBAMEncoder(nn.Module):
    def __init__(self, out_dim=256, dropout=0.3):
        super().__init__()

        self.backbone = timm.create_model(
            "convnext_tiny.fb_in22k_ft_in1k",
            pretrained=True,
            features_only=True,
            out_indices=(3,)
        )

        channels = self.backbone.feature_info.channels()[-1]

        self.cbam = CBAM(channels)
        self.pool = nn.AdaptiveAvgPool2d(1)

        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.LayerNorm(channels),
            nn.Dropout(dropout),
            nn.Linear(channels, out_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        feat = self.backbone(x)[-1]
        feat = self.cbam(feat)
        feat = self.pool(feat)
        feat = self.proj(feat)
        return feat

In [16]:
class PseudoSwarmBeamNet(nn.Module):
    def __init__(
        self,
        pos_dim,
        num_classes,
        embed_dim=256,
        num_heads=4,
        num_layers=2,
        dropout=0.3
    ):
        super().__init__()

        self.img_encoder = ConvNeXtCBAMEncoder(
            out_dim=embed_dim,
            dropout=dropout
        )

        self.pos_encoder = nn.Sequential(
            nn.Linear(pos_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(128, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU()
        )

        self.gate = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.Sigmoid()
        )

        self.fusion_proj = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 2,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.user_transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, imgs, pos):
        # imgs: [B, 5, 3, 224, 224]
        # pos : [B, 5, pos_dim]

        B, K, C, H, W = imgs.shape

        imgs_flat = imgs.reshape(B * K, C, H, W)
        img_feat = self.img_encoder(imgs_flat)
        img_feat = img_feat.reshape(B, K, -1)

        pos_feat = self.pos_encoder(pos)

        concat = torch.cat([img_feat, pos_feat], dim=-1)
        gate = self.gate(concat)

        fused = gate * img_feat + (1.0 - gate) * pos_feat

        fused = self.fusion_proj(torch.cat([fused, pos_feat], dim=-1))

        fused = self.user_transformer(fused)

        logits = self.classifier(fused)  # [B, 5, num_classes]

        return logits

In [17]:
model = PseudoSwarmBeamNet(
    pos_dim=pos_dim,
    num_classes=num_classes,
    embed_dim=256,
    num_heads=4,
    num_layers=2,
    dropout=0.3
).to(device)

with torch.no_grad():
    out = model(imgs_b.to(device), pos_b.to(device))

print(out.shape)  # [batch, 5, num_classes]

torch.Size([8, 5, 30])


In [20]:
def swarm_loss_fn(logits, labels, label_smoothing=0.05):
    B, K, C = logits.shape
    return F.cross_entropy(
        logits.reshape(B*K, C),
        labels.reshape(B*K),
        label_smoothing=label_smoothing
    )


@torch.no_grad()
def evaluate_swarm(model, loader, device, ks=(1, 2, 3, 5)):
    model.eval()

    total = 0
    correct = {k: 0 for k in ks}
    loss_sum = 0.0

    for imgs, pos, y in loader:
        imgs = imgs.to(device)
        pos = pos.to(device)
        y = y.to(device)

        logits = model(imgs, pos)

        loss = swarm_loss_fn(logits, y, label_smoothing=0.0)

        B, K, C = logits.shape
        logits_flat = logits.reshape(B*K, C)
        y_flat = y.reshape(B*K)

        max_k = max(ks)
        pred = logits_flat.topk(max_k, dim=1

SyntaxError: incomplete input (592327805.py, line 32)